### Business Analysis & Dashboard Datasets

Este notebook consolida as métricas de negócio utilizadas no dashboard executivo da LH Nautical.

A camada Gold funciona como a vitrine dos dados já preparados. A partir dela, este notebook cria indicadores e visões voltadas à tomada de decisão, sem alterar os dados das camadas anteriores.

Principais frentes:

- Performance comercial
- Clientes e portfólio
- Canais de venda
- Estoque e operações
- Forecast de demanda
- Sistema de recomendação

In [0]:
%sql
DESCRIBE nautical_lighthouse.gold.fact_sales;

In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_kpis AS

WITH order_costs AS (
    SELECT
        order_id,
        SUM(total_cost) AS total_cost,
        SUM(quantity) AS unidades
    FROM nautical_lighthouse.gold.fact_sales
    GROUP BY order_id
)

SELECT
    ROUND(SUM(o.total), 2) AS receita_total,
    COUNT(DISTINCT o.id) AS total_pedidos,

    ROUND(
        SUM(o.total) / COUNT(DISTINCT o.id),
        2
    ) AS ticket_medio,

    ROUND(
        SUM(o.total - oc.total_cost),
        2
    ) AS lucro_bruto,

    ROUND(
        SUM(o.total - oc.total_cost)
        / SUM(o.total) * 100,
        2
    ) AS margem_bruta_pct,

    SUM(oc.unidades) AS unidades_vendidas,
    COUNT(DISTINCT o.customer_id) AS clientes

FROM nautical_lighthouse.silver.orders o

JOIN order_costs oc
    ON oc.order_id = o.id

WHERE o.status <> 'cancelled';

In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_kpis;

In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_monthly_sales AS

SELECT
    DATE_TRUNC('month', placed_at) AS mes,
    ROUND(SUM(total), 2) AS receita,
    COUNT(DISTINCT id) AS pedidos
FROM nautical_lighthouse.silver.orders
WHERE status <> 'cancelled'
GROUP BY DATE_TRUNC('month', placed_at)
ORDER BY mes;

In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_monthly_sales
ORDER BY mes;

In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_channel_performance AS

SELECT
    channel,
    ROUND(SUM(total), 2) AS receita,
    COUNT(DISTINCT id) AS pedidos,
    ROUND(
        SUM(total) / COUNT(DISTINCT id),
        2
    ) AS ticket_medio
FROM nautical_lighthouse.silver.orders
WHERE status <> 'cancelled'
GROUP BY channel
ORDER BY receita DESC;

In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_channel_performance;

In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_weekday_sales AS

WITH limites AS (
    SELECT
        MIN(CAST(placed_at AS DATE)) AS data_inicial,
        MAX(CAST(placed_at AS DATE)) AS data_final
    FROM nautical_lighthouse.silver.orders
),

calendario AS (
    SELECT
        data,
        weekday(data) + 1 AS numero_dia_semana,
        CASE weekday(data)
            WHEN 0 THEN 'Segunda-feira'
            WHEN 1 THEN 'Terça-feira'
            WHEN 2 THEN 'Quarta-feira'
            WHEN 3 THEN 'Quinta-feira'
            WHEN 4 THEN 'Sexta-feira'
            WHEN 5 THEN 'Sábado'
            WHEN 6 THEN 'Domingo'
        END AS dia_semana
    FROM (
        SELECT EXPLODE(
            SEQUENCE(
                data_inicial,
                data_final,
                INTERVAL 1 DAY
            )
        ) AS data
        FROM limites
    )
),

vendas_diarias AS (
    SELECT
        CAST(placed_at AS DATE) AS data,
        SUM(total) AS total_vendas
    FROM nautical_lighthouse.silver.orders
    WHERE channel = 'pos'
      AND status <> 'cancelled'
    GROUP BY CAST(placed_at AS DATE)
),

calendario_com_vendas AS (
    SELECT
        c.data,
        c.numero_dia_semana,
        c.dia_semana,
        COALESCE(v.total_vendas, 0) AS total_vendas
    FROM calendario c
    LEFT JOIN vendas_diarias v
        ON v.data = c.data
)

SELECT
    numero_dia_semana,
    dia_semana,
    ROUND(AVG(total_vendas), 2) AS media_vendas
FROM calendario_com_vendas
GROUP BY
    numero_dia_semana,
    dia_semana;

In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_weekday_sales
ORDER BY media_vendas ASC;

In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_elite_customers AS

WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(total) AS faturamento_total,
        COUNT(id) AS frequencia,
        SUM(total) / COUNT(id) AS ticket_medio
    FROM nautical_lighthouse.silver.orders
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
),

customer_categories AS (
    SELECT
        customer_id,
        COUNT(DISTINCT category_name) AS diversidade_categorias
    FROM nautical_lighthouse.gold.fact_sales
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
)

SELECT
    cs.customer_id,
    ROUND(cs.faturamento_total, 2) AS faturamento_total,
    cs.frequencia,
    ROUND(cs.ticket_medio, 2) AS ticket_medio,
    cc.diversidade_categorias
FROM customer_sales cs
JOIN customer_categories cc
    ON cc.customer_id = cs.customer_id
WHERE cc.diversidade_categorias >= 13
ORDER BY
    cs.ticket_medio DESC,
    cs.customer_id ASC
LIMIT 10;

In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_elite_customers;

In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_elite_categories AS

SELECT
    fs.category_name,
    SUM(fs.quantity) AS total_itens,
    COUNT(DISTINCT fs.customer_id) AS clientes_elite
FROM nautical_lighthouse.gold.fact_sales fs
JOIN nautical_lighthouse.gold.dashboard_elite_customers ec
    ON ec.customer_id = fs.customer_id
GROUP BY fs.category_name
ORDER BY total_itens DESC;

In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_elite_categories
ORDER BY total_itens DESC;

In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_demand_forecast AS

WITH vendas_mensais AS (
    SELECT
        DATE_TRUNC('month', o.placed_at) AS mes,
        SUM(oi.quantity) AS vendas_reais
    FROM nautical_lighthouse.silver.orders o

    JOIN nautical_lighthouse.silver.order_items oi
        ON oi.order_id = o.id

    JOIN nautical_lighthouse.silver.product_variants pv
        ON pv.id = oi.product_variant_id

    JOIN nautical_lighthouse.silver.products p
        ON p.id = pv.product_id

    WHERE p.name = 'Bússola de Bordo 702'

    GROUP BY DATE_TRUNC('month', o.placed_at)
),

previsoes AS (
    SELECT
        mes,
        vendas_reais,

        AVG(vendas_reais) OVER (
            ORDER BY mes
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS previsao

    FROM vendas_mensais
)

SELECT
    mes,
    vendas_reais,
    ROUND(previsao, 2) AS previsao,
    ROUND(ABS(vendas_reais - previsao), 2) AS erro_absoluto

FROM previsoes

WHERE mes BETWEEN
    TIMESTAMP '2026-01-01'
    AND TIMESTAMP '2026-03-01'

ORDER BY mes;

In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_demand_forecast;

In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_forecast_kpis AS

SELECT
    ROUND(SUM(previsao)) AS previsao_total_q1,
    ROUND(AVG(erro_absoluto), 2) AS mae
FROM nautical_lighthouse.gold.dashboard_demand_forecast;

In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_forecast_kpis;

In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_product_recommendations AS

WITH interactions AS (
    SELECT DISTINCT
        fs.customer_id,
        dp.product_id,
        dp.product_name
    FROM nautical_lighthouse.gold.fact_sales fs
    JOIN nautical_lighthouse.gold.dim_product dp
        ON dp.product_variant_id = fs.product_variant_id
    WHERE fs.customer_id IS NOT NULL
),

reference_product AS (
    SELECT DISTINCT
        product_id
    FROM interactions
    WHERE product_name = 'Motor de Popa 1949'
),

product_counts AS (
    SELECT
        product_id,
        product_name,
        COUNT(DISTINCT customer_id) AS clientes_produto
    FROM interactions
    GROUP BY
        product_id,
        product_name
),

reference_count AS (
    SELECT
        COUNT(DISTINCT i.customer_id) AS clientes_referencia
    FROM interactions i
    JOIN reference_product r
        ON i.product_id = r.product_id
),

common_customers AS (
    SELECT
        i.product_id,
        i.product_name,
        COUNT(DISTINCT i.customer_id) AS clientes_em_comum
    FROM interactions i
    JOIN interactions ref
        ON i.customer_id = ref.customer_id
    JOIN reference_product r
        ON ref.product_id = r.product_id
    WHERE i.product_id <> r.product_id
    GROUP BY
        i.product_id,
        i.product_name
)

SELECT
    cc.product_id,
    cc.product_name,
    ROUND(
        cc.clientes_em_comum
        / SQRT(pc.clientes_produto * rc.clientes_referencia),
        6
    ) AS similarity
FROM common_customers cc
JOIN product_counts pc
    ON pc.product_id = cc.product_id
CROSS JOIN reference_count rc
ORDER BY similarity DESC
LIMIT 5;

In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_product_recommendations
ORDER BY similarity DESC;

In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_weekday_sales_detail AS

WITH limites AS (
    SELECT
        MIN(CAST(placed_at AS DATE)) AS data_inicial,
        MAX(CAST(placed_at AS DATE)) AS data_final
    FROM nautical_lighthouse.silver.orders
),

calendario AS (
    SELECT
        data,
        weekday(data) + 1 AS numero_dia_semana,
        CASE weekday(data)
            WHEN 0 THEN 'Segunda-feira'
            WHEN 1 THEN 'Terça-feira'
            WHEN 2 THEN 'Quarta-feira'
            WHEN 3 THEN 'Quinta-feira'
            WHEN 4 THEN 'Sexta-feira'
            WHEN 5 THEN 'Sábado'
            WHEN 6 THEN 'Domingo'
        END AS dia_semana
    FROM (
        SELECT EXPLODE(
            SEQUENCE(
                data_inicial,
                data_final,
                INTERVAL 1 DAY
            )
        ) AS data
        FROM limites
    )
),

vendas_diarias AS (
    SELECT
        CAST(placed_at AS DATE) AS data,
        SUM(total) AS total_vendas
    FROM nautical_lighthouse.silver.orders
    WHERE channel = 'pos'
      AND status <> 'cancelled'
    GROUP BY CAST(placed_at AS DATE)
),

base AS (
    SELECT
        c.data,
        c.numero_dia_semana,
        c.dia_semana,
        COALESCE(v.total_vendas, 0) AS total_vendas
    FROM calendario c
    LEFT JOIN vendas_diarias v
        ON v.data = c.data
),

por_dia_semana AS (
    SELECT
        numero_dia_semana,
        dia_semana,
        AVG(total_vendas) AS media_vendas,
        SUM(CASE WHEN total_vendas = 0 THEN 1 ELSE 0 END) AS dias_sem_venda
    FROM base
    GROUP BY numero_dia_semana, dia_semana
),

media_geral AS (
    SELECT AVG(total_vendas) AS media_geral
    FROM base
)

SELECT
    p.numero_dia_semana,
    p.dia_semana,
    ROUND(p.media_vendas, 2) AS media_vendas,
    p.dias_sem_venda,

    ROUND(
        p.media_vendas - m.media_geral,
        2
    ) AS diferenca_media,

    ROUND(
        ((p.media_vendas - m.media_geral) / m.media_geral) * 100,
        2
    ) AS diferenca_pct

FROM por_dia_semana p
CROSS JOIN media_geral m;

In [0]:

%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_weekday_sales_detail
ORDER BY media_vendas ASC;

In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_inventory_by_category AS

SELECT
    category_name,
    SUM(quantity_on_hand) AS unidades_estoque,
    ROUND(SUM(inventory_cost_value), 2) AS capital_imobilizado,
    ROUND(SUM(inventory_sale_value), 2) AS valor_potencial_venda
FROM nautical_lighthouse.gold.fact_inventory
GROUP BY category_name
ORDER BY capital_imobilizado DESC;

In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_inventory_by_category
ORDER BY capital_imobilizado DESC;

In [0]:

%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_returns_by_category AS

SELECT
    category_name,
    SUM(quantity) AS itens_devolvidos,
    ROUND(SUM(refund_amount), 2) AS valor_reembolsado,
    COUNT(DISTINCT return_id) AS devolucoes
FROM nautical_lighthouse.gold.fact_returns
WHERE return_status = 'completed'
GROUP BY category_name
ORDER BY valor_reembolsado DESC;

In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_returns_by_category
ORDER BY valor_reembolsado DESC;


In [0]:
%sql
CREATE OR REPLACE VIEW nautical_lighthouse.gold.dashboard_inventory_risk AS

WITH estoque AS (
    SELECT
        category_name,
        SUM(quantity_on_hand) AS unidades_estoque,
        SUM(inventory_cost_value) AS capital_imobilizado
    FROM nautical_lighthouse.gold.fact_inventory
    GROUP BY category_name
),

vendas_2025 AS (
    SELECT
        category_name,
        SUM(quantity) AS unidades_vendidas_2025
    FROM nautical_lighthouse.gold.fact_sales
    WHERE order_year = 2025
      AND is_cancelled = false
    GROUP BY category_name
)

SELECT
    e.category_name,
    ROUND(e.unidades_estoque, 2) AS unidades_estoque,
    ROUND(e.capital_imobilizado, 2) AS capital_imobilizado,
    COALESCE(v.unidades_vendidas_2025, 0) AS unidades_vendidas_2025,

    ROUND(
        e.capital_imobilizado /
        NULLIF(v.unidades_vendidas_2025, 0),
        2
    ) AS capital_por_unidade_vendida

FROM estoque e
LEFT JOIN vendas_2025 v
    ON v.category_name = e.category_name;


In [0]:
%sql
SELECT *
FROM nautical_lighthouse.gold.dashboard_inventory_risk
ORDER BY capital_por_unidade_vendida DESC;
